<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
LangSmith Evaluations Basics
</b></font> </br></p>

---


**Beitrag zum Leitprojekt:** `sources_evaluator` prüft, ob die erwartete Datei abgerufen wurde. `not_im_corpus_evaluator` sucht einen Ablehnungsmarker bei fehlender Evidenz. Diese Textprüfungen helfen, auffällige Antworten für die fachliche Prüfung auszuwählen. Ob eine Antwort durch die Quelle belegt ist, muss zusätzlich geprüft werden. Eine Tool-Wahl-Metrik ist eine mögliche Erweiterung.


**Praxis-Transfer:** Die Evaluation prüft, ob Maras Briefing-Fragen mit passenden Quellen beantwortet werden und fehlende Informationen korrekt als unsicher oder nicht belegt erscheinen.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "false"
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"] = "m13_langsmith_evaluations"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    WORKER,
    CODING,
    JUDGE,
    PLANNER,
    WORKER_PREMIUM,
    FRONTIER,
    IMAGE_GENERATION,
    TRANSCRIPTION,
    TRANSCRIPTION_SEGMENTS,
    EMBEDDINGS,
)
# LangSmith Tracing
run_cfg = {
    "tags": ["langsmith"],
    "metadata": {"notebook": "langsmith_evaluations_basics", "version": "1.0"}
}

In [ ]:
#@title 📂 Google Drive mounten und Eval-Sets laden { display-mode: "form" }
import json
import shutil
from pathlib import Path
from google.colab import drive
from genai_lib.utilities import copy_from_github

# Die Wissensdatenbank (Chroma) liegt aus dem Modul RAG-Agent auf Google Drive
drive.mount("/content/drive")

EVAL_TARGET = "/content/eval"
shutil.rmtree(EVAL_TARGET, ignore_errors=True)
copy_from_github(
    source="ralf-42/Agenten/02_daten/05_sonstiges",
    target=EVAL_TARGET,
    mask="eval_pmo*.json",
)

eval_files = sorted(Path(EVAL_TARGET).glob("eval_pmo*.json"))
mprint(f"Eval-Dateien: {', '.join(p.name for p in eval_files)}")

# 1 | Übersicht
---


Dieses Modul macht die Qualität des PMO-Assistenten messbar. Der Fokus liegt nicht mehr auf einzelnen Demo-Fragen, sondern auf wiederholbaren Testfällen.

⚠️ **Voraussetzung:** RAG-Agenten muss vorher einmal durchgelaufen sein. Dieses Modul bewertet die dort aufgebaute Wissensdatenbank (Chroma-Collection `pmo_korpus` auf Google Drive) und baut keinen eigenen Index.

**Ziel des Moduls**

1. Wissensdatenbank öffnen sowie `eval_pmo.json` und `eval_pmo_edge.json` laden
2. Retrieval-Treffer gegen erwartete Quellen und Begriffe prüfen
3. Antworten anhand einfacher, transparenter Metriken bewerten
4. Ein LangSmith-Dataset für spätere Regressionstests (Test auf unbeabsichtigte Nebenwirkungen) vorbereiten

**Metriken im Modul**

| Metrik | Zweck |
|---|---|
| Quellen-Treffer | Erwartete Quelldatei unter den Top-k-Treffern? |
| Begriff-Abdeckung | Erwartete Begriffe in Treffertext oder Antwort vorhanden? |
| Out-of-Corpus-Check | Enthält die Antwort bei fehlender Evidenz den Marker `Nicht im Korpus`? |
| Falsche-Ablehnung-Check | Enthält die Antwort auf eine beantwortbare Frage den Ablehnungsmarker? |
| LangSmith-Experiment | Reproduzierbarer Evaluationslauf mit Scores |


In [ ]:
#@markdown   <p><font size="4" color='green'>  Evaluation Workflow</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    EVAL["📋 Eval-Set<br><small>eval_pmo*.json</small>"] --> RET["🔎 Retriever<br><small>Top-k Chunks</small>"]
    RET --> MET["📏 Retrieval-Metriken<br><small>Quelle + Begriffe</small>"]
    RET --> RAG["🤖 RAG-Antwort"]
    RAG --> ANS["📏 Antwort-Metriken<br><small>Begriffe + Nicht im Korpus</small>"]
    MET --> REPORT["📊 Bericht"]
    ANS --> REPORT
    REPORT --> LS["LangSmith Dataset<br><small>Regression</small>"]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class EVAL input;
    class MET,ANS metric;
    class REPORT output;
    class LS target;
'''

mermaid(diagram, width=1100)


# 2 | Eval-Set laden
---


In [ ]:
# Funktionen
def load_eval_set(filename: str):
    file_path = Path(EVAL_TARGET) / filename
    with file_path.open("r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
# Evaluations-Sets laden
main_eval = load_eval_set("eval_pmo.json")
edge_eval = load_eval_set("eval_pmo_edge.json")
all_eval_cases = [
    {**case, "set": "main"} for case in main_eval
] + [
    {**case, "set": "edge"} for case in edge_eval
]

mprint(f"Haupt-Eval: {len(main_eval)} Fragen")
mprint(f"Edge-Eval:  {len(edge_eval)} Fragen")
mprint(f"Gesamt:     {len(all_eval_cases)} Fragen")
mprint()
for case in all_eval_cases[:5]:
    mprint(f"{case['id']} · {case['typ']} · {case['frage']}")


In [ ]:
# Tabelle mit den Eval-Testfällen erstellen.
from collections import Counter

types = Counter(case["typ"] for case in all_eval_cases)
sets = Counter(case["set"] for case in all_eval_cases)

rows = ["| Kategorie | Anzahl |", "|---|---:|"]
for name, count in sorted(sets.items()):
    rows.append(f"| Set: {name} | {count} |")
for name, count in sorted(types.items()):
    rows.append(f"| Typ: {name} | {count} |")

mprint("\n".join(rows))

# 3 | Wissensdatenbank aus Modul RAG-Agent nutzen

Die Evaluation läuft auf der in **RAG-Agent** aufgebauten Chroma-Collection `pmo_korpus` (Google Drive). Es wird kein eigener Index erstellt — so bewertet das Modul genau die Datenbasis, die der RAG-Agent verwendet.

📦 **Voraussetzung:** RAG-Agent muss einmal durchgelaufen sein, damit die Collection auf Google Drive existiert. `genai_lib.pmo_rag.get_pmo_vectorstore()` öffnet sie ohne Neuaufbau und bricht mit einer klaren Meldung ab, wenn Google Drive nicht gemountet ist.

⚠️ **Hinweis:** Die RAG-Agent-Chunks sind zeichenbasiert (300 Zeichen). Die Scorer prüfen daher auf Top-k-Ebene, ob erwartete Quellen und Begriffe vorkommen.

In [ ]:
import re

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

from genai_lib.pmo_rag import get_pmo_vectorstore

In [ ]:
# LLM initialisieren
llm = init_chat_model(JUDGE)

In [ ]:
# Collection öffnen (kein Neuaufbau)
vectorstore = get_pmo_vectorstore()
count = vectorstore._collection.count()
assert count > 0, "Collection leer — zuerst Modul RAG-Agent ausführen"

In [ ]:
# aktuelles Modul vergleicht k=3 mit k=5 und nutzt k=5 für die Antwortstichprobe.
RETRIEVAL_K = 5
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
mprint(f"Collection-Größe: {count} (Antwort-Retrieval k={RETRIEVAL_K})")

# 4 | Retrieval-Bewertung
---


Im Rahmen der Evaluation des PMO-Assistenten wird die Qualität der Informationsbeschaffung (Retrieval) anhand von zwei zentralen Dimensionen gemessen:

1. **Quelle (Quellen-Treffer / Source Hit):**
   * **Was es bedeutet:** Diese Metrik prüft, ob das korrekte Quelldokument (z. B. `projektauftrag_kompass.pdf`) in den vom Retriever zurückgelieferten Top-$k$ Textstücken (Chunks) enthalten ist.
   * **Warum es wichtig ist:** Ein RAG-System kann nur dann eine sachlich richtige und belegbare Antwort generieren, wenn der Retriever im ersten Schritt überhaupt die richtige Informationsquelle aus der Vektordatenbank heraussucht.

2. **Begriffe (Begriff-Abdeckung / Concept Coverage):**
   * **Was es bedeutet:** Hierbei wird überprüft, ob wichtige, vordefinierte Schlüsselbegriffe (z. B. spezifische Projektabkürzungen wie `R2`, Personennamen oder Statusangaben) in den abgerufenen Textstücken vorkommen.
   * **Warum es wichtig ist:** Selbst wenn das richtige Dokument gefunden wird, müssen die konkreten Chunks auch die präzisen Details enthalten, die zur Beantwortung der Frage notwendig sind. Die Begriff-Abdeckung ist ein Textindikator: Sie beweist weder die Relevanz einer Passage noch die korrekte Bedeutung der gefundenen Begriffe.

In [ ]:
# Funktionen

def normalize_text(text: str) -> str:
    # Tausenderpunkte entfernen ("124.500" -> "124500"), Datumsangaben bleiben unberührt
    return re.sub(r"(?<=\d)\.(?=\d{3}\b)", "", (text or "").lower())


def sources_name(doc) -> str:
    return doc.metadata.get("source", "?").split("/")[-1]


def term_groups(case: dict):
    groups = case.get("erwartete_begriffgruppen")
    if groups:
        return groups
    # Rollenbezeichnungen gelten als Varianten; Personennamen bleiben unverändert.
    aliases = {"projektleitung": ["Projektleitung", "Projektleiterin", "Projektleiter"]}
    return [aliases.get(normalize_text(term), [term])
            for term in case.get("erwartete_begriffe", [])]


def evaluate_term_groups(text: str, case: dict):
    text_normalized = normalize_text(text)
    groups = term_groups(case)
    groups_matches = []
    for group in groups:
        matches = [term for term in group if normalize_text(term) in text_normalized]
        if matches:
            groups_matches.append(matches[0])
    score = len(groups_matches) / len(groups) if groups else 1.0
    return score, groups_matches


def evaluate_retrieval(case: dict, k: int = 3) -> dict:
    docs = vectorstore.similarity_search(case["frage"], k=k)
    sources = [sources_name(doc) for doc in docs]
    text = "\n".join(doc.page_content for doc in docs)
    expected_source = case.get("erwartete_quelle")
    # Out-of-corpus-Fälle sind für den Quellen-Treffer nicht anwendbar.
    source_hit = None if expected_source is None else expected_source in sources
    if expected_source is None:
        term_score, term_matches = None, []
    else:
        term_score, term_matches = evaluate_term_groups(text, case)
    return {
        "id": case["id"], "set": case["set"], "typ": case["typ"],
        "frage": case["frage"], "erwartete_quelle": expected_source,
        "quellen": sources, "quelle_getroffen": source_hit,
        "begriff_score": None if term_score is None else round(term_score, 3),
        "begriff_treffer": term_matches,
    }


def summarize_retrieval(results):
    # Quellenquote nur für Fälle mit definierter Ground Truth berechnen.
    source_values = [r["quelle_getroffen"] for r in results if r["quelle_getroffen"] is not None]
    source_accuracy = sum(source_values) / len(source_values) if source_values else float("nan")
    term_values = [r["begriff_score"] for r in results if r["begriff_score"] is not None]
    term_mean = sum(term_values) / len(term_values) if term_values else float("nan")
    return source_accuracy, term_mean, len(source_values)


In [ ]:
# Retrieval k=3 & k=5
retrieval_results_k3 = [evaluate_retrieval(case, k=3) for case in all_eval_cases]
retrieval_results_k5 = [evaluate_retrieval(case, k=5) for case in all_eval_cases]
source_accuracy_k3, term_mean_k3, n_sources_k3 = summarize_retrieval(retrieval_results_k3)
source_accuracy_k5, term_mean_k5, n_sources_k5 = summarize_retrieval(retrieval_results_k5)

# Die weiteren Antworttests verwenden bewusst den breiteren k=5-Kontext.
retrieval_results = retrieval_results_k5
mprint(
    f"Retrieval-Vergleich: k=3 Quelle {source_accuracy_k3:.2%} (n={n_sources_k3}), "
    f"Begriffe {term_mean_k3:.2%} | "
    f"k=5 Quelle {source_accuracy_k5:.2%} (n={n_sources_k5}), "
    f"Begriffe {term_mean_k5:.2%}"
)


In [ ]:
import pandas as pd

# Quelldateien deduplizieren, da mehrere Chunks derselben Datei vorkommen können.
df_retrieval = pd.DataFrame([
    {
        "ID": r["id"],
        "Set": r["set"],
        "Quelle getroffen": "n. a." if r["quelle_getroffen"] is None else int(r["quelle_getroffen"]),
        "Begriff-Score": r["begriff_score"] if r["begriff_score"] is not None else float("nan"),
        "Top 5 Dateien (eindeutig)": ", ".join(list(dict.fromkeys(r["quellen"]))[:5]),
    }
    for r in retrieval_results
])

mprint("## Retrieval-Baseline (k=5)")
mprint("Quellen-Trefferquote: nur Fälle mit erwarteter Quelle; n. a. = Out-of-Corpus-Fall.")
display(df_retrieval.head(12))


# 5 | Einfache Antwortbewertung
---


Die Antwortbewertung prüft einfache Textmerkmale der generierten RAG-Antwort. Sie liefert Hinweise für die fachliche Prüfung, keine automatische Bestätigung der Richtigkeit.

1. **Begriff-Abdeckung:** Erwartete Begriffe oder explizite Aliasgruppen werden gesucht. Bei Referenzen mit „Projektleitung“ zählen auch „Projektleiterin“ und „Projektleiter“. Ob die genannte Person tatsächlich zuständig ist, prüft diese Metrik nicht.
2. **Ablehnungsmarker bei fehlender Evidenz:** Für Fälle ohne erwartete Quelle wird nach „Nicht im Korpus“ gesucht. `1` bedeutet nur, dass dieser Wortlaut vorkommt; `0` bedeutet, dass er fehlt. Eine anders formulierte Ablehnung kann fachlich richtig sein. Auch eine Antwort mit diesem Marker kann anschließend unbelegte Aussagen enthalten.
3. **Ablehnungsmarker bei beantwortbaren Fragen:** Der LangSmith-Score `falsche_ablehnung_vermieden` ist `0`, wenn der Marker vorkommt, sonst `1`. Teilantworten mit einer benannten Evidenzlücke und abweichende Formulierungen müssen manuell beurteilt werden.

Die Stichprobe zeigt daher zusätzlich jede vollständige Antwort, eindeutige Retrieval-Quellen und nicht gefundene Begriffsgruppen. Die Quellenliste zeigt abgerufene Dateien, nicht automatisch die vom Modell zitierten oder korrekt verwendeten Quellen.


In [ ]:
# Prompt
rag_prompt = PromptTemplate.from_template(
    "Antwort ausschließlich auf Basis des Kontexts formulieren. "
    "Enthält der Kontext relevante Informationen, beantworte die Frage damit, "
    "auch wenn sie nur teilweise abgedeckt ist (z. B. Status 'in Arbeit' statt "
    "eines Abschlusstermins), und benenne, was fehlt. "
    "Nur wenn der Kontext keinerlei relevante Information enthält, antworte "
    "ausschließlich mit: Nicht im Korpus. Das gilt auch, wenn der Kontext das Thema "
    "nur erwähnt, die erfragte Information (z. B. medizinische oder rechtliche "
    "Beratung) aber nicht enthält. Nichts raten oder ergänzen.\n\n"
    "Kontext:\n{context}\n\n"
    "Frage: {question}\n\n"
    "Antwort mit Quellenhinweis:"
)

In [ ]:
# Funktionen zur Formatierung
def format_docs(docs):
    parts = []
    for doc in docs:
        source = sources_name(doc)
        parts.append(f"Quelle: {source}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

In [ ]:
# Chain definieren
rag_chain = rag_prompt | llm | StrOutputParser()

In [ ]:
# RAG abrufen
def answer_with_rag(question: str, run_name: str):
    """Führt die drei RAG-Schritte explizit und genau einmal aus."""
    # 1. Dokumente explizit abrufen
    retrieved = retriever.invoke(
        question,
        config={**run_cfg, "run_name": f"m13_{run_name}_retrieval"},
    )
    # 2. Dokumente formatieren
    context_text = format_docs(retrieved)
    # 3. Chain mit dem fertigen Dictionary aufrufen
    answer = rag_chain.invoke(
        {"context": context_text, "question": question},
        config={**run_cfg, "run_name": f"m13_{run_name}_antwort"},
    )
    return retrieved, answer

def answer_eval_case(case: dict) -> dict:
    docs, answer = answer_with_rag(
        case["frage"], "kap5_answer_eval_case"
    )
    return {
        "id": case["id"],
        "frage": case["frage"],
        "antwort": answer,
        "quellen": [sources_name(doc) for doc in docs],
    }

In [ ]:
# Antwort-RAG vs. Eval bewerten

def evaluate_answer(case: dict, answer: str) -> dict:
    expected_not_im_corpus = case.get("erwartete_quelle") is None
    answer_normalized = normalize_text(answer)
    if expected_not_im_corpus:
        not_im_corpus_ok = "nicht im korpus" in answer_normalized
        term_score = 1.0 if not_im_corpus_ok else 0.0
        term_matches = ["Nicht im Korpus"] if not_im_corpus_ok else []
    else:
        # Für beantwortbare Fälle ist die Ablehnungsmetrik nicht anwendbar.
        not_im_corpus_ok = None
        term_score, term_matches = evaluate_term_groups(answer, case)
    return {
        "id": case["id"],
        "antwort_begriff_score": round(term_score, 3),
        "nicht_im_korpus_ok": not_im_corpus_ok,
        "antwort_begriff_treffer": term_matches,
    }


In [ ]:
# Kleine Stichprobe für Kostenkontrolle im Kurslauf.
eval_sample = main_eval[:5] + edge_eval[:3]
eval_sample = [
    {**case, "set": "main"} if case["id"].startswith("q") else {**case, "set": "edge"}
    for case in eval_sample
]

answer_results = []
for case in eval_sample:
    output = answer_eval_case(case)
    scores = evaluate_answer(case, output["antwort"])
    answer_results.append({**output, **scores})
    mprint(f"**{case['id']}:** Textcheck-Score {scores['antwort_begriff_score']:.2f}")

mprint(f"**Antwort-Stichprobe:** {len(answer_results)} Fälle")

In [ ]:
# Übersicht und vollständige Antworten ohne neue Modellaufrufe anzeigen
import pandas as pd

sample_by_id = {case["id"]: case for case in eval_sample}
df_answers = pd.DataFrame([
    {
        "ID": r["id"],
        "Textcheck-Score": round(r["antwort_begriff_score"], 2),
        "Prüfart": "Ablehnungsmarker" if r["nicht_im_korpus_ok"] is not None else "Begriff-Abdeckung",
        "Ablehnungsmarker gefunden": ("n. a." if r["nicht_im_korpus_ok"] is None else int(r["nicht_im_korpus_ok"])),
    }
    for r in answer_results
])
mprint("## Antwort-Baseline (Stichprobe): Textprüfungen")
display(df_answers)
for result in answer_results:
    case = sample_by_id[result["id"]]
    mprint(f"### {result['id']}: {result['frage']}")
    if case.get("erwartete_quelle") is not None:
        _, matches = evaluate_term_groups(result["antwort"], case)
        missing = [group for group in term_groups(case) if not any(term in matches for term in group)]
        mprint(f"Nicht gefundene Begriffsgruppen: {missing or 'keine'}")
    else:
        mprint("Ablehnungsfall: Wortlaut und fachliche Angemessenheit getrennt prüfen.")
    sources = list(dict.fromkeys(result["quellen"]))
    mprint(f"Abgerufene Quelldateien ({len(sources)}): {', '.join(sources) or 'keine'}")
    mprint(f"**Vollständige Antwort:**\n\n{result['antwort']}")


# 6 | LangSmith-Dataset vorbereiten
---


Dasselbe Eval-Set kann als LangSmith-Dataset angelegt und mit `evaluate()` ausgeführt werden. Das Anlegen oder Prüfen des Datasets führt noch keine Evaluation aus.

Der Lauf liefert fünf Score-Keys:

- `quelle_top5`: erwartete Datei unter den fünf abgerufenen Chunks. Dies prüft weder ein Zitat in der Antwort noch dessen Belegbarkeit.
- `begriff_abdeckung`: Anteil gefundener Begriffsgruppen bei beantwortbaren Fragen; bei Fällen ohne erwartete Quelle wird aus Kompatibilitätsgründen stattdessen der binäre Ablehnungsmarker erfasst. Diese Fallgruppen getrennt auswerten.
- `nicht_im_korpus_ablehnung`: Marker „Nicht im Korpus“ bei Fällen ohne erwartete Quelle gefunden.
- `falsche_ablehnung_vermieden`: Marker bei beantwortbaren Fällen nicht gefunden. Dies ist keine Garantie für eine korrekte Antwort.
- `quelle_nicht_anwendbar`: technischer Platzhalter mit Wert 1.0 für Fälle ohne erwartete Quelle; nicht als Qualitätswert mitteln.

Nicht zutreffende Evaluatoren erscheinen in der Ergebnistabelle als `NaN`. Ohne vollständigen Evaluationslauf zeigt die Diagnose nur einen Hinweis. Die lokalen Antworten aus Kapitel 5 bleiben unabhängig davon einsehbar.


In [ ]:
from langsmith import Client

client = Client()
DATASET_NAME = "m13_briefing_eval_v1"

In [ ]:
# Liste aufbauen
langsmith_examples = [
    {
        "inputs": {"frage": case["frage"], "id": case["id"], "set": case["set"]},
        "outputs": {
            "antwort": case["erwartete_antwort_kurz"],
            "erwartete_quelle": case.get("erwartete_quelle"),
            "erwartete_begriffe": case.get("erwartete_begriffe", []),
            "erwartete_begriffgruppen": case.get("erwartete_begriffgruppen"),
        },
    }
    for case in all_eval_cases
]

In [ ]:
# Eval-Set bereits vorhanden oder neun anlegen?
existing = [ds.name for ds in client.list_datasets()]
if DATASET_NAME in existing:
    dataset = next(ds for ds in client.list_datasets() if ds.name == DATASET_NAME)
    mprint(f"Dataset wiedergefunden: {DATASET_NAME}")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="PMO-Assistent Eval-Set mit Hauptfragen und Edge-Cases",
    )
    for ex in langsmith_examples:
        client.create_example(
            dataset_id=dataset.id,
            inputs=ex["inputs"],
            outputs=ex["outputs"],
        )
    mprint(f"Dataset erstellt: {DATASET_NAME}")

In [ ]:
# Eval-Set lokal vs. LangSmith auf Anzahl, IDs und Inhalte vergleichen
my_dataset_name = DATASET_NAME
examples_in_dataset = list(client.list_examples(dataset_id=dataset.id))
mprint(f"Examples im Dataset: {len(examples_in_dataset)}")
local_by_id = {ex["inputs"]["id"]: ex for ex in langsmith_examples}
dataset_by_id = {ex.inputs.get("id"): ex for ex in examples_in_dataset}
ids_match = set(dataset_by_id) == set(local_by_id) and len(examples_in_dataset) == len(local_by_id)
contents_match = ids_match and all(
    dataset_by_id[case_id].inputs == expected["inputs"]
    and dataset_by_id[case_id].outputs == expected["outputs"]
    for case_id, expected in local_by_id.items()
)
if not contents_match:
    mprint("Dataset weicht bei Anzahl, IDs oder Inhalten vom lokalen Eval-Set ab.")
    mprint("Dataset in LangSmith aktualisieren oder neu anlegen, bevor evaluate() ausgeführt wird.")
else:
    mprint("Dataset stimmt bei Anzahl, IDs und Inhalten mit dem lokalen Eval-Set überein.")


In [ ]:
# Funktionen für die Evalkuation mit LangSmith
from langsmith.evaluation import evaluate

def briefing_target(inputs: dict) -> dict:
    docs, answer = answer_with_rag(
        inputs["frage"], "kap6_briefing_target"
    )
    return {
        "antwort": answer,
        "quellen": [sources_name(doc) for doc in docs],
    }

def sources_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    expected_source = reference_outputs.get("erwartete_quelle")
    sources = outputs.get("quellen", [])
    if expected_source is None:
        return {"key": "quelle_nicht_anwendbar", "score": 1.0}
    score = 1.0 if expected_source in sources else 0.0
    return {"key": "quelle_top5", "score": score}

def term_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    answer = outputs.get("antwort", "")
    if reference_outputs.get("erwartete_quelle") is None:
        # Ablehnungs-Fall: nur die Ablehnung zählt, nicht die Begriffsliste
        score = 1.0 if "nicht im korpus" in normalize_text(answer) else 0.0
        return {"key": "begriff_abdeckung", "score": score}
    groups = reference_outputs.get("erwartete_begriffgruppen")
    if groups:
        case = {"erwartete_begriffgruppen": groups}
    else:
        case = {"erwartete_begriffe": reference_outputs.get("erwartete_begriffe", [])}
    score, _ = evaluate_term_groups(answer, case)
    return {"key": "begriff_abdeckung", "score": round(score, 3)}

def not_im_corpus_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    expected_source = reference_outputs.get("erwartete_quelle")
    answer = normalize_text(outputs.get("antwort", ""))
    if expected_source is not None:
        # Beantwortbarer Fall: eine Ablehnung wäre eine falsche Ablehnung
        score = 0.0 if "nicht im korpus" in answer else 1.0
        return {"key": "falsche_ablehnung_vermieden", "score": score}
    score = 1.0 if "nicht im korpus" in answer else 0.0
    return {"key": "nicht_im_korpus_ablehnung", "score": score}

In [ ]:
# Aufruf komplette Evaluation
# Hinweis Trace-Limit: evaluate() erzeugt pro Dataset-Beispiel mindestens einen Trace (Anzahl aus dem gewählten Dataset) plus Evaluator-Runs
RUN_FULL_EVALUATION = False  # Für einen bewussten LangSmith-Lauf auf True setzen.
EVAL_SUBSET = None           # None = alle Beispiele, "main" oder "edge" = nur dieser Teil des Datasets
my_eval_results = None  # Ergebnisse eines früheren Laufs nicht versehentlich weiterverwenden.
if RUN_FULL_EVALUATION:
    if not contents_match:
        raise ValueError("Dataset stimmt nicht mit den lokalen Referenzen überein; zuerst aktualisieren.")
    # Auswahl clientseitig über das Feld "set" in den Inputs (unabhängig von Dataset-Splits)
    eval_examples = [
        ex for ex in client.list_examples(dataset_id=dataset.id)
        if EVAL_SUBSET is None or ex.inputs.get("set") == EVAL_SUBSET
    ]
    mprint(f"Evaluation startet mit {len(eval_examples)} Beispielen (Subset: {EVAL_SUBSET or 'alle'})")

    my_eval_results = evaluate(
        briefing_target,
        data=eval_examples,
        evaluators=[sources_evaluator, term_evaluator, not_im_corpus_evaluator],
        experiment_prefix="m13_briefing_baseline",
        max_concurrency=1,
        disable_evaluator_tracing=True,  # Evaluator-Runs nicht als eigene Traces speichern (spart Kontingent)
    )

    # Scores auch direkt im Notebook anzeigen.
    display(my_eval_results.to_pandas())
else:
    mprint("Vollständige LangSmith-Evaluation deaktiviert. Für einen neuen Lauf RUN_FULL_EVALUATION = True setzen.")


In [ ]:
# Diagnose: vollständige Antworten eines tatsächlich ausgeführten LangSmith-Laufs
CASE_IDS = None  # None = auffällige Fälle; alternativ z. B. ["e01", "q01"]
THRESHOLD = 1.0

if globals().get("my_eval_results") is None:
    mprint("Keine LangSmith-Evaluation vorhanden. Zuerst den Lauf in Kapitel 6 aktivieren und ausführen. Lokale Antworten stehen in Kapitel 5.")
else:
    df_eval = my_eval_results.to_pandas()
    if df_eval.empty:
        mprint("Die Evaluation enthält keine Ergebnisse.")
    else:
        score_columns = [name for name in (
            "feedback.quelle_top5", "feedback.begriff_abdeckung",
            "feedback.nicht_im_korpus_ablehnung", "feedback.falsche_ablehnung_vermieden",
        ) if name in df_eval.columns]
        if CASE_IDS is not None:
            selection = df_eval[df_eval["inputs.id"].isin(CASE_IDS)]
        else:
            selection = df_eval[df_eval[score_columns].lt(THRESHOLD).any(axis=1)] if score_columns else df_eval
        mprint(f"{len(selection)} von {len(df_eval)} Fällen ausgewählt")
        for _, row in selection.sort_values("inputs.id").iterrows():
            mprint("---")
            mprint(f"**{row['inputs.id']}** | Textchecks: {row[score_columns].dropna().to_dict()}")
            mprint(f"**Frage:** {row['inputs.frage']}")
            mprint(f"**Abgerufene Quellen:** {list(dict.fromkeys(row['outputs.quellen']))}")
            mprint(f"**Vollständige Antwort:**\n\n{row['outputs.antwort']}")
        mprint("Niedrige Textscores sind Prüfhinweise. Ablehnungsvarianten, Teilantworten und Quellenbelege fachlich beurteilen.")


<p><font color='darkblue' size="4">📌 <b>Einordnung</b></font></p>

Die Evaluatoren messen **Quellentreffer, Begriffsabdeckung und Ablehnungsmarker**. Sie prüfen weder die Belegbarkeit einzelner Aussagen noch eine echte Eskalation oder einen Human-in-the-Loop-Schritt.

**Grenzen des Scorings:** Die Begriffssuche erkennt keine Verneinung. „Mara Vogt ist nicht zuständig“ kann trotzdem den Namenscheck erfüllen. Die Varianten „Projektleitung“, „Projektleiterin“ und „Projektleiter“ verhindern unnötige Abzüge bei Rollenbezeichnungen, ersetzen aber keine inhaltliche Prüfung. Für belastbare Aussagen sind zusätzliche Quellenreviews oder ein validierter LLM-Judge nötig.

**Ablehnungsfälle:** Bei fehlender erwarteter Quelle misst `begriff_abdeckung` den Wortlaut „Nicht im Korpus“, keine Begriffabdeckung. Ein Wert von 0 belegt keine Halluzination: Eine sinngemäße Ablehnung kann anders formuliert sein. Ein Wert von 1 schließt unbelegte Zusätze nicht aus. Diese Werte nur innerhalb der Fallgruppe vergleichen.

**Beantwortbare Fälle:** `falsche_ablehnung_vermieden` sucht ebenfalls nur diesen Wortlaut. Eine Teilantwort kann die verfügbaren Fakten korrekt nennen und zugleich eine Informationslücke markieren. Ob das angemessen ist, muss anhand der vollständigen Frage, Antwort und Quelle beurteilt werden.

**Historische Befunde:** In einem früheren Lauf war e14 wegen des fehlenden Abschlusstermins abgelehnt worden, obwohl der Status „in Arbeit“ vorlag. Der Prompt erlaubt deshalb belegte Teilantworten mit benannten Lücken. Bei q34 verhinderte die Schreibweise „124.500“ einen Treffer für „124500“; `normalize_text` vereinheitlicht diese Zahlenform. Diese Beispiele dokumentieren frühere Beobachtungen, keine Ergebnisse des aktuellen Laufs.

**Ausführung:** Nach Änderungen an Prompt oder Scoring die betroffenen Zellen neu ausführen. Gespeicherte Ergebnisse müssen zum aktuellen Code passen. Eine deaktivierte LangSmith-Evaluation liefert keine Experimentergebnisse; das Dataset allein reicht dafür nicht aus.


# 7 | Datenschutz und Trace-Grenzen
---


LangSmith-Tracing überträgt Inputs und Outputs an den LangSmith-Server. Der PMO-Korpus enthält ausschließlich fiktive Projektunterlagen; echte personenbezogene oder vertrauliche Daten gehören trotzdem nicht in Eval-Sets oder Traces.

| Risiko | Gegenmaßnahme |
|---|---|
| Personenbezogene Daten im Trace | Vor dem Aufruf filtern oder anonymisieren |
| Vermischung von Entwicklung und Produktion | Separate LangSmith-Projekte verwenden |
| Unklare Ground Truth | Eval-Set versionieren und Quellen pflegen |
| Out-of-Corpus-Fragen | `Nicht im Korpus` explizit bewerten, bei beantwortbaren Fragen als falsche Ablehnung werten |


In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t
_t.sleep(2)
show_trace(os.environ["LANGSMITH_PROJECT"], limit=3, show_steps=True)

# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
- Die ersten 10 Fälle aus `all_eval_cases` auswählen (`eval_auswahl`).
- `evaluate_retrieval(case, k=...)` einmal mit `k=1` und einmal mit `k=5` ausführen.
- Die Ergebnisse in `meine_retrieval_k1` und `meine_retrieval_k5` speichern.
- Für beide Varianten Quellen-Trefferquote und mittleren Begriff-Score berechnen und vergleichen.


In [ ]:
# Grundlagen: Retrieval mit k=1 und k=5 vergleichen
# Startpunkt: evaluate_retrieval(case, k) und all_eval_cases aus den Kapiteln 2 und 4

# 1. eval_auswahl = all_eval_cases[:10]
# 2. meine_retrieval_k1 und meine_retrieval_k5 mit evaluate_retrieval(case, k=...) berechnen
# 3. Quellen-Trefferquote und mittleren Begriff-Score pro Variante ausgeben und vergleichen


**Aufbau**
- Fünf **andere** Hauptfälle (`main_eval[5:10]`, also q06 bis q10) und drei weitere Edge-Cases (`edge_eval[3:6]`, also e04 bis e06) mit `answer_eval_case()` beantworten.
- Mit `evaluate_answer()` Begriff-Abdeckung und `Nicht im Korpus`-Verhalten bewerten.
- Die Ergebnisse in `meine_antwort_ergebnisse` speichern.


In [ ]:
# Aufbau: Antwortbewertung für andere Fälle
# Startpunkt: answer_eval_case(), evaluate_answer(), main_eval und edge_eval aus den Kapiteln 2 und 5

# 1. meine_stichprobe = main_eval[5:10] + edge_eval[3:6]
# 2. Für jeden Fall: Antwort erzeugen (answer_eval_case) und bewerten (evaluate_answer)
# 3. Ergebnisse in meine_antwort_ergebnisse sammeln, Begriff-Score und Nicht-im-Korpus-Ergebnis pro Fall ausgeben


**Vertiefung**
- LangSmith-Dataset aus `DATASET_NAME` verwenden.
- `evaluate()` mit Quellen-, Begriffs- und Out-of-Corpus-Evaluator ausführen.
- Ergebnis in `meine_eval_results` speichern und Score-Spalten prüfen.


In [ ]:
# Vertiefung: LangSmith-Evaluation ausführen
# Startpunkt: evaluate(), briefing_target(), die drei Evaluatoren aus Kapitel 6

# 1. evaluate() mit data=mein_dataset_name und den drei Evaluatoren aufrufen
# 2. Ergebnis in meine_eval_results speichern
# 3. Score-Spalten in meine_eval_results.to_pandas() prüfen

**Praxis-Transfer-Aufgabe: PMO-Assistent**

Eval-Fragen für Meeting- und Briefings anlegen.

1. Welche reale Arbeitsfrage löst **dieser** Notebook-Baustein im PMO-Assistent?
2. Welche Eingabedaten oder Dokumente braucht er?
3. Welche Ausgabe sollte er liefern?
4. Welche Risiken oder Grenzen bleiben?
5. Wann wäre Human Review nötig?
6. Ist der passende Lösungsweg hier Prompt, strukturierte Ausgabe, RAG, Tool, Agent oder Workflow?

<p><font color='darkblue' size="4">
 <b>✨ Viz</b>
</font></p>

- [RAG-Pipeline](https://ralf-42.github.io/Agenten/visualisierungen/rag_prozess_visualisierung/)
- [Token-Verbrauch & Caching](https://ralf-42.github.io/Agenten/visualisierungen/token_verbrauch_caching/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [LangSmith](https://ralf-42.github.io/Agenten/05-frameworks/langsmith.html)
- [Einsteiger LangSmith](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langsmith.html)
- [Evaluation & Observability](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/evaluation-observability.html)
- [LangSmith Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langsmith-best-practices.html)
- [Agent Evaluation & Observability Best Practices](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-evaluation-observability-best-practices.html)
- [Agent Harness](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-harness.html)
